# EA4 — Transición Energética y Análisis del Consumo de Vehículos Eléctricos en Ciudades Inteligentes

**Autor:** Fredy Ramírez

**Pregunta de investigación:**

¿Cómo varían la energía consumida y las sesiones de recarga de vehículos eléctricos?

**Objetivo general:**

Analizar el comportamiento de las sesiones de recarga de vehículos eléctricos a partir de los registros de energía consumida, duración y potencia nominal de las estaciones, identificando diferencias temporales y geográficas que permitan comprender los patrones de utilización de la infraestructura de recarga.

**Objetivos específicos:**

1. Examinar la distribución de las sesiones de recarga según la hora del día.
2. Comparar la energía consumida y la potencia media observada entre ciudades.
3. Evaluar las diferencias entre los tipos de conectores disponibles.
4. Identificar registros con posibles inconsistencias entre energía, duración y potencia nominal.
5. Representar los resultados mediante visualizaciones que faciliten su interpretación.

**Alcance y limitaciones:**

El dataset contiene información de fecha y hora, ciudad, estación, potencia nominal, duración de las sesiones, tipo de conector y energía consumida. No incluye temperatura ambiental, identificación de vehículos, capacidad de batería ni autonomía. Por esta razón, el análisis se limita a las variables disponibles y no pretende medir directamente la eficiencia energética de los vehículos ni proyectar su autonomía.

## Prompt para la creación del dashboard

Actúa como un asistente experto en análisis de datos, Python, Pandas, NumPy y Streamlit. Ayúdame a desarrollar un proyecto de análisis de datos sobre el consumo energético de las sesiones de recarga de vehículos eléctricos.

El objetivo es responder la siguiente pregunta:

¿Cómo varían la energía consumida y la potencia media observada en las sesiones de recarga?

Utiliza las siguientes variables del dataset:
- Fecha y hora de la sesión.
- Ciudad.
- Identificador de estación.
- Potencia nominal en kW.
- Duración de la sesión en minutos.
- Energía consumida en kWh.
- Tipo de conector.

Quiero que me ayudes a:

1. Revisar la calidad de los datos, identificando valores nulos, duplicados, registros inválidos e inconsistencias.
2. Proponer una limpieza de datos justificada, sin eliminar automáticamente valores sospechosos.
3. Calcular la potencia media observada de cada sesión y compararla con la potencia nominal.
4. Analizar patrones horarios, diferencias entre ciudades y tipos de conectores.
5. Crear visualizaciones claras e interactivas.
6. Diseñar un dashboard en Streamlit con filtros que actualicen los resultados.
7. Obtener hallazgos cuantitativos y redactar conclusiones basadas en los datos.
8. Verificar los resultados mediante cálculos independientes.
9. Indicar las limitaciones del dataset y evitar conclusiones que los datos no permitan demostrar.

No inventes resultados. Si no dispones de los datos, Explica el código y las decisiones metodológicas para que pueda comprenderlas y revisarlas antes de incorporarlas al proyecto.


## 1. Configuración e importaciones

Ejecuta las celdas en orden. En el terminal, crea y activa un entorno virtual e instala las dependencias desde `requirements.txt`.


| Elemento | Descripción |
| :--- | :--- |
| **Fuente** | Hugging Face – Global EV Charging Station Telemetry |
| **Enlace** | [ProWiddan/Global-EV-Charging-Station-Telemetry](https://huggingface.co) |
| **Tipo de datos** | Registros de sesiones de recarga |
| **Registro** | Cada fila representa una sesión de recarga |
| `timestamp` | Fecha y hora de la sesión |
| `city` | Ciudad donde se registra la recarga |
| `station_id` | Identificador de la estación |
| `power_kw` | Potencia nominal de la estación |
| `session_duration_mins` | Duración de la sesión en minutos |
| `energy_consumed_kwh` | Energía consumida durante la sesión |
| `connector_type` | Tipo de conector utilizado |



| N.º | Criterio de evaluación | Máximo | Autoevaluación |
| :---: | :--- | :---: | :---: |
| 1 | Planteamiento del problema, pregunta de investigación y procedencia de los datos | 20 | 16 |
| 2 | Preparación, limpieza y validación de los datos | 20 | 17 |
| 3 | Funcionamiento, visualización e interacción del dashboard | 25 | 15 |
| 4 | Presentación de resultados, hallazgos numéricos y verificación | 25 | 18 |
| 5 | Uso de inteligencia artificial, prompt y evaluación de sus propuestas | 10 | 6 |
| | **TOTAL** | **100** | **72** |


In [184]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import os

DATA_DIR = Path('datos')
DATA_DIR.mkdir(exist_ok=True)
DATASET_URL = 'https://huggingface.co/datasets/ProWiddan/Global-EV-Charging-Station-Telemetry'
print('Origen del dataset:', DATASET_URL)

Origen del dataset: https://huggingface.co/datasets/ProWiddan/Global-EV-Charging-Station-Telemetry


## 2. Ingesta de datos desde Hugging Face

Esta celda intenta localizar archivos de datos CSV/JSON/Parquet del repositorio del dataset. Si el repositorio requiere una configuración distinta, revisaremos el listado de archivos y adaptaremos el cargador.

In [185]:
# Instala huggingface_hub si aún no está disponible: pip install huggingface_hub
from huggingface_hub import list_repo_files, hf_hub_download

dataset_repo = 'ProWiddan/Global-EV-Charging-Station-Telemetry'
files = list_repo_files(dataset_repo, repo_type='dataset')
data_files = [f for f in files if f.lower().endswith(('.csv', '.json', '.parquet'))]
print('Archivos de datos encontrados:')
for f in data_files:
    print('-', f)

if not data_files:
    raise FileNotFoundError('No se encontraron CSV, JSON o Parquet en el repositorio. Revisa el listado de archivos del dataset.')

chosen_file = data_files[0]
local_file = hf_hub_download(repo_id=dataset_repo, filename=chosen_file, repo_type='dataset', local_dir=DATA_DIR)
print('Archivo descargado:', local_file)

Archivos de datos encontrados:
- ev_charging_telemetry.csv
Archivo descargado: /workspaces/analitica/EA4_Actividad_IA/datos/ev_charging_telemetry.csv


In [186]:
# Carga según extensión
ext = Path(chosen_file).suffix.lower()
if ext == '.csv':
    df = pd.read_csv(local_file)
elif ext == '.json':
    df = pd.read_json(local_file)
elif ext == '.parquet':
    df = pd.read_parquet(local_file)
else:
    raise ValueError(f'Formato no admitido: {ext}')

display(df.head())
print('Dimensiones (filas, columnas):', df.shape)

,timestamp,station_id,city,power_kw,session_duration_mins,connector_type,energy_consumed_kwh
0,2025-08-28 04:20:25.163578,ST-995,Oslo,150.0,118,CCS,72.82
1,2025-08-28 06:20:25.163578,ST-939,Amsterdam,150.0,141,Tesla Supercharger,137.69
2,2025-08-28 07:20:25.163578,ST-292,Oslo,22.0,87,Type2,11.18
3,2025-08-28 07:20:25.163578,ST-247,London,350.0,113,Tesla Supercharger,25.38
4,2025-08-28 10:20:25.163578,ST-453,London,350.0,162,CCS,139.86


Dimensiones (filas, columnas): (5000, 7)


## 3. Auditoría inicial de datos

Antes de limpiar, examinamos nombres de columnas, tipos, faltantes y duplicados. No reemplazaremos valores faltantes por cero salvo que el significado del dato justifique esa decisión.

In [187]:
display(pd.DataFrame({'tipo': df.dtypes.astype(str), 'nulos': df.isnull().sum(), 'nulos_%': (df.isnull().mean()*100).round(2)}))
print('Filas duplicadas:', df.duplicated().sum())
print('Columnas disponibles:', df.columns.tolist())

,tipo,nulos,nulos_%
timestamp,str,0,0.0
station_id,str,0,0.0
city,str,0,0.0
power_kw,float64,0,0.0
session_duration_mins,int64,0,0.0
connector_type,str,0,0.0
energy_consumed_kwh,float64,0,0.0


Filas duplicadas: 0
Columnas disponibles: ['timestamp', 'station_id', 'city', 'power_kw', 'session_duration_mins', 'connector_type', 'energy_consumed_kwh']


## 4. Próximas etapas

- Identificar campos de temperatura, fecha/hora, energía, duración, potencia, vehículo y batería.
- Normalizar unidades y documentar datos faltantes.
- Crear variables derivadas con Pandas y NumPy (solo si existen los campos necesarios).
- Segmentar con `.query()`, agrupar con `groupby()` y verificar cálculos mediante máscaras NumPy.
- Construir visualizaciones y dashboard de Streamlit.
- Redactar hallazgos numéricos y decisiones operativas basados en resultados reales.
- Exportar el resumen de auditoría a TXT.

In [188]:
# Convertir timestamp a formato datetime
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

# Comprobar el resultado
print("Tipo de dato de timestamp:", df['timestamp'].dtype)
print("Primeras fechas:")
display(df[['timestamp']].head())

# Comprobar si la conversión produjo valores inválidos
print("Fechas inválidas:", df['timestamp'].isna().sum())

Tipo de dato de timestamp: datetime64[us]
Primeras fechas:


,timestamp
0,2025-08-28 04:20:25.163578
1,2025-08-28 06:20:25.163578
2,2025-08-28 07:20:25.163578
3,2025-08-28 07:20:25.163578
4,2025-08-28 10:20:25.163578


Fechas inválidas: 0


In [189]:
# Extraer información temporal
df['hora'] = df['timestamp'].dt.hour
df['dia'] = df['timestamp'].dt.day
df['mes'] = df['timestamp'].dt.month
df['dia_semana'] = df['timestamp'].dt.day_name()

# Mostrar las nuevas columnas
display(df[['timestamp', 'hora', 'dia', 'mes', 'dia_semana']].head(10))

# Comprobar rango temporal del dataset
print("Fecha inicial:", df['timestamp'].min())
print("Fecha final:", df['timestamp'].max())

# Horas disponibles
print("Horas registradas:", sorted(df['hora'].unique()))

,timestamp,hora,dia,mes,dia_semana
0,2025-08-28 04:20:25.163578,4,28,8,Thursday
1,2025-08-28 06:20:25.163578,6,28,8,Thursday
2,2025-08-28 07:20:25.163578,7,28,8,Thursday
3,2025-08-28 07:20:25.163578,7,28,8,Thursday
4,2025-08-28 10:20:25.163578,10,28,8,Thursday
5,2025-08-28 12:20:25.163578,12,28,8,Thursday
6,2025-08-28 14:20:25.163578,14,28,8,Thursday
7,2025-08-28 16:20:25.163578,16,28,8,Thursday
8,2025-08-28 18:20:25.163578,18,28,8,Thursday
9,2025-08-28 20:20:25.163578,20,28,8,Thursday


Fecha inicial: 2025-08-28 04:20:25.163578
Fecha final: 2026-08-28 01:20:25.163578
Horas registradas: [np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12), np.int32(13), np.int32(14), np.int32(15), np.int32(16), np.int32(17), np.int32(18), np.int32(19), np.int32(20), np.int32(21), np.int32(22), np.int32(23)]


In [190]:
# Estadísticas descriptivas de las variables numéricas
display(df[['power_kw', 'session_duration_mins', 'energy_consumed_kwh']].describe().round(2))

# Número de ciudades registradas
print("Número de ciudades:", df['city'].nunique())
print("\nCiudades:")
print(df['city'].value_counts())

# Tipos de conectores y frecuencia
print("\nTipos de conectores:")
print(df['connector_type'].value_counts())

# Número de estaciones distintas
print("\nNúmero de estaciones:", df['station_id'].nunique())

,power_kw,session_duration_mins,energy_consumed_kwh
count,5000.00,5000.00,5000.00
mean,139.27,102.86,72.93
std,127.37,55.01,39.13
min,22.00,10.00,4.03
25%,22.00,55.00,39.33
50%,50.00,101.00,73.53
75%,150.00,151.00,106.68
max,350.00,199.00,139.99


Número de ciudades: 7

Ciudades:
city
Dubai        748
Oslo         733
London       722
Stockholm    715
Berlin       697
Tokyo        696
Amsterdam    689
Name: count, dtype: int64

Tipos de conectores:
connector_type
CHAdeMO               1280
Type2                 1254
CCS                   1253
Tesla Supercharger    1213
Name: count, dtype: int64

Número de estaciones: 890


In [191]:

# Calcular la potencia media observada durante cada sesión (kW)

df['potencia_media_observada_kw'] = np.where(
    (df['session_duration_mins'] > 0) &
    (df['energy_consumed_kwh'] >= 0),
    df['energy_consumed_kwh'] / (df['session_duration_mins'] / 60),
    np.nan
)

# Comparar la potencia media observada con la potencia nominal
df['ratio_potencia'] = np.where(
    df['power_kw'] > 0,
    df['potencia_media_observada_kw'] / df['power_kw'],
    np.nan
)

# Mostrar los resultados
display(
    df[[
        'power_kw',
        'session_duration_mins',
        'energy_consumed_kwh',
        'potencia_media_observada_kw',
        'ratio_potencia'
    ]].head(10).round(2)
)

# Estadísticas descriptivas
display(
    df[[
        'potencia_media_observada_kw',
        'ratio_potencia'
    ]].describe().round(2)
)


,power_kw,session_duration_mins,energy_consumed_kwh,potencia_media_observada_kw,ratio_potencia
0,150.0,118,72.82,37.03,0.25
1,150.0,141,137.69,58.59,0.39
2,22.0,87,11.18,7.71,0.35
3,350.0,113,25.38,13.48,0.04
4,350.0,162,139.86,51.80,0.15
5,22.0,64,100.14,93.88,4.27
6,50.0,162,59.57,22.06,0.44
7,350.0,19,31.11,98.24,0.28
8,50.0,83,38.65,27.94,0.56
9,22.0,135,93.20,41.42,1.88


,potencia_media_observada_kw,ratio_potencia
count,5000.00,5000.00
mean,71.56,1.41
std,91.90,2.77
min,1.47,0.00
25%,22.85,0.17
50%,42.37,0.51
75%,80.52,1.48
max,813.72,36.99


### Análisis de la potencia media observada y detección de anomalías

A partir de las variables de potencia nominal del cargador (`power_kw`), duración de la sesión (`session_duration_mins`) y energía consumida (`energy_consumed_kwh`), se calculó la potencia media observada de cada sesión, convirtiendo previamente la duración de minutos a horas. También se obtuvo un ratio entre la potencia media calculada y la potencia nominal del cargador para evaluar la coherencia de los registros.

Los resultados muestran una potencia media observada de **71,56 kW**, mientras que la mediana es de **42,37 kW**, lo que indica una distribución asimétrica influida por valores elevados. El ratio de potencia presenta una media de **1,41** y un máximo de **36,99**, lo que evidencia la existencia de registros potencialmente anómalos. En algunos casos, la energía consumida y el tiempo de carga implican potencias medias superiores a la potencia nominal del cargador, algo que requiere revisión.

Estas inconsistencias pueden deberse a errores de registro, datos sintéticos, diferencias en la interpretación de las variables o problemas de calidad del dataset. Por tanto, no se consideran automáticamente valores válidos de funcionamiento real.

En conclusión, este análisis demuestra la importancia de validar la coherencia entre las variables energéticas antes de realizar comparaciones o extraer conclusiones sobre el comportamiento de las estaciones de recarga. Las anomalías identificadas se conservarán inicialmente para su estudio y documentación, evitando eliminarlas sin una justificación metodológica.


## 5. Limpieza y validación de datos

En esta etapa se comprueba la coherencia de los registros utilizados en el análisis. Se revisan valores negativos, duraciones no válidas y posibles inconsistencias entre la potencia media observada y la potencia nominal de las estaciones.

Los registros sospechosos no se eliminan automáticamente. Se identifican y documentan para evitar perder información sin una justificación.


In [192]:

# Crear una copia para trabajar con los datos
df_analisis = df.copy()

# Comprobar valores inválidos en las variables principales
print("Duraciones iguales o inferiores a cero:",
      (df_analisis['session_duration_mins'] <= 0).sum())

print("Energías negativas:",
      (df_analisis['energy_consumed_kwh'] < 0).sum())

print("Potencias nominales no válidas:",
      (df_analisis['power_kw'] <= 0).sum())

# Identificar registros con potencia media superior
# a la potencia nominal del cargador
df_analisis['posible_anomalia'] = (
    df_analisis['ratio_potencia'] > 1
)

print(
    "Registros con potencia media superior a la nominal:",
    df_analisis['posible_anomalia'].sum()
)

# Mostrar los registros sospechosos
anomalias = df_analisis.query('posible_anomalia == True')

display(
    anomalias[[
        'city',
        'connector_type',
        'power_kw',
        'session_duration_mins',
        'energy_consumed_kwh',
        'potencia_media_observada_kw',
        'ratio_potencia'
    ]].head(10).round(2)
)


Duraciones iguales o inferiores a cero: 0
Energías negativas: 0
Potencias nominales no válidas: 0
Registros con potencia media superior a la nominal: 1691


,city,connector_type,power_kw,session_duration_mins,energy_consumed_kwh,potencia_media_observada_kw,ratio_potencia
5,Tokyo,CCS,22.0,64,100.14,93.88,4.27
9,Oslo,Type2,22.0,135,93.20,41.42,1.88
10,Stockholm,Tesla Supercharger,22.0,88,135.41,92.32,4.20
11,Stockholm,CCS,50.0,27,117.50,261.11,5.22
15,Tokyo,Tesla Supercharger,22.0,35,129.61,222.19,10.10
16,Tokyo,CCS,50.0,22,63.73,173.81,3.48
19,Stockholm,CHAdeMO,22.0,114,108.74,57.23,2.60
20,Dubai,Tesla Supercharger,22.0,65,94.10,86.86,3.95
21,Dubai,CCS,22.0,94,89.12,56.89,2.59
25,Dubai,Tesla Supercharger,22.0,106,113.79,64.41,2.93



## 6. Análisis temporal de las sesiones de recarga

Se analiza cómo varían las sesiones de recarga a lo largo de las 24 horas del día. Para ello se agrupan los registros por hora y se calculan el número de sesiones, la energía consumida y la potencia media observada.

Este análisis permite identificar los periodos de mayor actividad y observar posibles diferencias en el comportamiento de la infraestructura de recarga.


In [193]:

# Agrupar los registros por hora del día

analisis_horario = (
    df_analisis
    .groupby('hora')
    .agg(
        numero_sesiones=('timestamp', 'count'),
        energia_media_kwh=('energy_consumed_kwh', 'mean'),
        energia_total_kwh=('energy_consumed_kwh', 'sum'),
        duracion_media_min=('session_duration_mins', 'mean'),
        potencia_media_kw=('potencia_media_observada_kw', 'mean')
    )
    .reset_index()
)

# Ordenar las horas de menor a mayor
analisis_horario = analisis_horario.sort_values('hora')

display(analisis_horario.round(2))


,hora,numero_sesiones,energia_media_kwh,energia_total_kwh,duracion_media_min,potencia_media_kw
0,0,256,72.10,18456.52,104.25,76.43
1,1,197,70.02,13794.05,101.60,67.42
2,2,189,72.22,13650.45,99.19,70.17
3,3,191,78.16,14928.99,104.73,78.62
4,4,208,76.17,15842.79,102.29,73.44
5,5,186,64.49,11995.71,105.94,56.66
6,6,208,78.55,16338.35,105.98,71.56
7,7,230,71.99,16556.62,103.66,73.56
8,8,187,75.85,14184.06,104.18,70.00
9,9,198,73.73,14597.96,98.69,73.57


In [194]:

# Hora con mayor número de sesiones
hora_mayor_actividad = analisis_horario.loc[
    analisis_horario['numero_sesiones'].idxmax()
]

# Hora con mayor consumo energético medio
hora_mayor_energia = analisis_horario.loc[
    analisis_horario['energia_media_kwh'].idxmax()
]

# Hora con mayor potencia media observada
hora_mayor_potencia = analisis_horario.loc[
    analisis_horario['potencia_media_kw'].idxmax()
]

print("Hora de mayor actividad:")
display(hora_mayor_actividad)

print("Hora de mayor energía media por sesión:")
display(hora_mayor_energia)

print("Hora de mayor potencia media observada:")
display(hora_mayor_potencia)


Hora de mayor actividad:


hora                      0.000000
numero_sesiones         256.000000
energia_media_kwh        72.095781
energia_total_kwh     18456.520000
duracion_media_min      104.250000
potencia_media_kw        76.425448
Name: 0, dtype: float64

Hora de mayor energía media por sesión:


hora                     23.000000
numero_sesiones         201.000000
energia_media_kwh        80.359801
energia_total_kwh     16152.320000
duracion_media_min      103.915423
potencia_media_kw        76.032730
Name: 23, dtype: float64

Hora de mayor potencia media observada:


hora                     18.000000
numero_sesiones         221.000000
energia_media_kwh        72.843167
energia_total_kwh     16098.340000
duracion_media_min      100.466063
potencia_media_kw        80.155026
Name: 18, dtype: float64


## 7. Comparación del comportamiento de recarga entre ciudades

Se comparan las sesiones de recarga registradas en las diferentes ciudades del dataset. Se estudian el volumen de sesiones, la energía consumida y la potencia media observada.

El propósito es identificar diferencias entre las ciudades y analizar cómo se comporta la infraestructura de recarga en cada una.


In [195]:

# Agrupar los datos por ciudad

analisis_ciudades = (
    df_analisis
    .groupby('city')
    .agg(
        numero_sesiones=('timestamp', 'count'),
        energia_media_kwh=('energy_consumed_kwh', 'mean'),
        energia_total_kwh=('energy_consumed_kwh', 'sum'),
        duracion_media_min=('session_duration_mins', 'mean'),
        potencia_media_kw=('potencia_media_observada_kw', 'mean')
    )
    .reset_index()
)

# Ordenar por energía media consumida
analisis_ciudades = analisis_ciudades.sort_values(
    'energia_media_kwh',
    ascending=False
)

display(analisis_ciudades.round(2))


,city,numero_sesiones,energia_media_kwh,energia_total_kwh,duracion_media_min,potencia_media_kw
0,Amsterdam,689,74.24,51148.63,100.25,79.11
1,Berlin,697,73.58,51286.88,104.26,71.23
4,Oslo,733,72.79,53351.47,107.00,67.68
6,Tokyo,696,72.69,50595.39,102.31,70.27
2,Dubai,748,72.63,54328.79,103.14,68.06
3,London,722,72.63,52435.41,101.26,72.80
5,Stockholm,715,72.00,51480.84,101.65,72.26


In [196]:

# Ciudad con mayor energía media por sesión
ciudad_mayor_energia = analisis_ciudades.loc[
    analisis_ciudades['energia_media_kwh'].idxmax()
]

# Ciudad con mayor número de sesiones
ciudad_mayor_actividad = analisis_ciudades.loc[
    analisis_ciudades['numero_sesiones'].idxmax()
]

print("Ciudad con mayor energía media por sesión:")
display(ciudad_mayor_energia)

print("Ciudad con mayor número de sesiones:")
display(ciudad_mayor_actividad)


Ciudad con mayor energía media por sesión:


city                   Amsterdam
numero_sesiones              689
energia_media_kwh      74.236038
energia_total_kwh       51148.63
duracion_media_min    100.251089
potencia_media_kw      79.107839
Name: 0, dtype: object

Ciudad con mayor número de sesiones:


city                       Dubai
numero_sesiones              748
energia_media_kwh      72.632072
energia_total_kwh       54328.79
duracion_media_min    103.139037
potencia_media_kw      68.056233
Name: 2, dtype: object


## 8. Análisis por tipo de conector

Se estudia la relación entre el tipo de conector utilizado y las características de las sesiones de recarga. Se comparan el número de sesiones, la energía media consumida, la duración y la potencia media observada.

Este análisis permite identificar diferencias entre las tecnologías de conexión presentes en el dataset.


In [197]:

# Agrupar por tipo de conector

analisis_conectores = (
    df_analisis
    .groupby('connector_type')
    .agg(
        numero_sesiones=('timestamp', 'count'),
        energia_media_kwh=('energy_consumed_kwh', 'mean'),
        energia_total_kwh=('energy_consumed_kwh', 'sum'),
        duracion_media_min=('session_duration_mins', 'mean'),
        potencia_media_kw=('potencia_media_observada_kw', 'mean')
    )
    .reset_index()
)

# Ordenar por potencia media observada
analisis_conectores = analisis_conectores.sort_values(
    'potencia_media_kw',
    ascending=False
)

display(analisis_conectores.round(2))


,connector_type,numero_sesiones,energia_media_kwh,energia_total_kwh,duracion_media_min,potencia_media_kw
3,Type2,1254,71.97,90246.19,101.72,72.44
2,Tesla Supercharger,1213,75.03,91017.34,104.95,72.16
1,CHAdeMO,1280,72.06,92242.18,102.52,71.47
0,CCS,1253,72.72,91121.70,102.34,70.19


In [198]:

# Conector con mayor energía media por sesión
conector_mayor_energia = analisis_conectores.loc[
    analisis_conectores['energia_media_kwh'].idxmax()
]

# Conector con mayor potencia media observada
conector_mayor_potencia = analisis_conectores.loc[
    analisis_conectores['potencia_media_kw'].idxmax()
]

print("Conector con mayor energía media:")
display(conector_mayor_energia)

print("Conector con mayor potencia media observada:")
display(conector_mayor_potencia)


Conector con mayor energía media:


connector_type        Tesla Supercharger
numero_sesiones                     1213
energia_media_kwh              75.034905
energia_total_kwh               91017.34
duracion_media_min            104.948063
potencia_media_kw              72.160323
Name: 2, dtype: object

Conector con mayor potencia media observada:


connector_type             Type2
numero_sesiones             1254
energia_media_kwh      71.966659
energia_total_kwh       90246.19
duracion_media_min    101.721691
potencia_media_kw      72.443748
Name: 3, dtype: object


## 9. Filtrado y clasificación de sesiones

Se aplican filtros para estudiar subconjuntos de sesiones de recarga. Además, se utiliza NumPy para clasificar los registros según la potencia media observada.

Estas operaciones permiten segmentar los datos y facilitar la comparación de diferentes comportamientos.


In [199]:

# Filtrar sesiones con consumo superior a 100 kWh
sesiones_alto_consumo = df_analisis.query(
    'energy_consumed_kwh > 100'
)

print("Sesiones con consumo superior a 100 kWh:",
      len(sesiones_alto_consumo))

display(
    sesiones_alto_consumo[[
        'city',
        'connector_type',
        'energy_consumed_kwh',
        'session_duration_mins'
    ]].head(10)
)

# Clasificar las sesiones por potencia media observada
df_analisis['categoria_potencia'] = np.select(
    [
        df_analisis['potencia_media_observada_kw'] < 22,
        df_analisis['potencia_media_observada_kw'] < 50,
        df_analisis['potencia_media_observada_kw'] >= 50
    ],
    [
        'Baja',
        'Media',
        'Alta'
    ],
    default='Sin clasificar'
)

# Contar las sesiones por categoría
print("Distribución por categoría de potencia:")
display(
    df_analisis['categoria_potencia']
    .value_counts()
    .to_frame('numero_sesiones')
)


Sesiones con consumo superior a 100 kWh: 1517


,city,connector_type,energy_consumed_kwh,session_duration_mins
1,Amsterdam,Tesla Supercharger,137.69,141
4,London,CCS,139.86,162
5,Tokyo,CCS,100.14,64
10,Stockholm,Tesla Supercharger,135.41,88
11,Stockholm,CCS,117.50,27
13,Oslo,CHAdeMO,135.47,196
15,Tokyo,Tesla Supercharger,129.61,35
19,Stockholm,CHAdeMO,108.74,114
22,Oslo,CHAdeMO,137.86,98
23,Berlin,Tesla Supercharger,105.17,87


Distribución por categoría de potencia:


,numero_sesiones
categoria_potencia,
Alta,2065
Media,1734
Baja,1201


## 10. Visualización de resultados

Se utilizan gráficos interactivos para representar los patrones de consumo energético, la actividad de las sesiones de recarga y las diferencias entre ciudades y conectores.

Las visualizaciones permiten identificar tendencias y comparar los resultados de forma más clara.

In [200]:

%pip install plotly

import plotly.express as px
import plotly.graph_objects as go


Note: you may need to restart the kernel to use updated packages.


In [201]:
%pip install "nbformat>=4.2.0"

Note: you may need to restart the kernel to use updated packages.


In [202]:
import nbformat

print("Versión de nbformat:", nbformat.__version__)

Versión de nbformat: 5.11.1


In [203]:
import sys
import nbformat
import plotly
import plotly.io as pio
import plotly.io as pio

pio.renderers.default = "iframe"

print("Python:", sys.executable)
print("nbformat:", nbformat.__version__)
print("Ruta nbformat:", nbformat.__file__)
print("Plotly:", plotly.__version__)
print("Renderer:", pio.renderers.default)

Python: /workspaces/analitica/.venv312/bin/python
nbformat: 5.11.1
Ruta nbformat: /workspaces/analitica/.venv312/lib/python3.12/site-packages/nbformat/__init__.py
Plotly: 7.1.0
Renderer: iframe


In [204]:
import plotly.io as pio

pio.renderers.default = "iframe"

In [205]:
import plotly.io as pio

pio.renderers.default = "plotly_mimetype"

In [206]:
%pip install --upgrade "nbformat>=4.2.0"


Note: you may need to restart the kernel to use updated packages.


In [207]:
#Número de sesiones por hora
fig = px.bar(
    analisis_horario,
    x='hora',
    y='numero_sesiones',
    title='Número de sesiones de recarga por hora',
    labels={
        'hora': 'Hora del día',
        'numero_sesiones': 'Número de sesiones'
    },
    text='numero_sesiones'
)

fig.update_layout(
    xaxis=dict(dtick=1),
    template='plotly_white'
)

fig.show()


In [208]:
#Comparación de energía media entre ciudades
fig = px.line(
    analisis_horario,
    x='hora',
    y='energia_media_kwh',
    markers=True,
    title='Energía media consumida por hora',
    labels={
        'hora': 'Hora del día',
        'energia_media_kwh': 'Energía media (kWh)'
    }
)

fig.update_layout(
    xaxis=dict(dtick=1),
    template='plotly_white'
)

fig.show()


In [209]:
#Potencia media observada por tipo de conector


fig = px.bar(
    analisis_conectores,
    x='connector_type',
    y='potencia_media_kw',
    color='connector_type',
    title='Potencia media observada según el tipo de conector',
    labels={
        'connector_type': 'Tipo de conector',
        'potencia_media_kw': 'Potencia media observada (kW)'
    },
    text_auto='.2f'
)

fig.update_layout(
    template='plotly_white',
    showlegend=False
)

fig.show()


In [210]:
#Relación entre duración y energía


fig = px.scatter(
    df_analisis,
    x='session_duration_mins',
    y='energy_consumed_kwh',
    color='connector_type',
    title='Relación entre duración y energía consumida',
    labels={
        'session_duration_mins': 'Duración (minutos)',
        'energy_consumed_kwh': 'Energía consumida (kWh)',
        'connector_type': 'Tipo de conector'
    },
    opacity=0.65
)

fig.update_layout(template='plotly_white')

fig.show()




## 11. Resultados y conclusiones

### 11.1. Comportamiento temporal

Se analizaron las sesiones de recarga agrupadas por hora. La mayor actividad se registró a las [HORA] horas, con [NÚMERO] sesiones.

En cuanto al consumo energético medio, el periodo destacado fue [HORA], con un promedio de [VALOR] kWh por sesión.

### 11.2. Comparación entre ciudades

La ciudad con mayor energía media consumida por sesión fue [CIUDAD], con [VALOR] kWh.

La ciudad con mayor número de sesiones fue [CIUDAD], con [NÚMERO] registros.

Estos resultados permiten diferenciar el volumen de utilización de la energía consumida por sesión.

### 11.3. Comparación de conectores

El conector con mayor energía media por sesión fue [CONECTOR], con [VALOR] kWh.

El conector con mayor potencia media observada fue [CONECTOR], con [VALOR] kW.

### 11.4. Calidad de los datos

Se identificaron [NÚMERO] registros en los que la potencia media observada superó la potencia nominal de la estación.

Estas inconsistencias deben tenerse en cuenta al interpretar los resultados, ya que pueden afectar las comparaciones entre ciudades y conectores.

### 11.5. Conclusión general

El análisis permitió identificar patrones temporales y diferencias en el comportamiento de las sesiones de recarga según la ciudad y el tipo de conector.

Los resultados muestran la utilidad de analizar conjuntamente la energía consumida, la duración y la potencia nominal para comprender el uso de la infraestructura de recarga.

Sin embargo, el dataset no contiene variables de temperatura, capacidad de batería ni autonomía. Por tanto, no es posible establecer conclusiones sobre la influencia de la temperatura ambiental o la autonomía real de los vehículos.


In [211]:

# Crear un informe de resultados

informe = []

informe.append("EA4 - ANALISIS DE RECARGA DE VEHICULOS ELECTRICOS")
informe.append("=" * 55)

informe.append(f"\nTotal de registros analizados: {len(df_analisis)}")
informe.append(f"Ciudades analizadas: {df_analisis['city'].nunique()}")
informe.append(f"Estaciones analizadas: {df_analisis['station_id'].nunique()}")
informe.append(f"Tipos de conectores: {df_analisis['connector_type'].nunique()}")

informe.append("\nRESULTADOS TEMPORALES")
informe.append("-" * 30)

informe.append(
    f"Hora de mayor actividad: "
    f"{int(hora_mayor_actividad['hora'])}:00"
)

informe.append(
    f"Sesiones registradas: "
    f"{int(hora_mayor_actividad['numero_sesiones'])}"
)

informe.append("\nRESULTADOS POR CIUDAD")
informe.append("-" * 30)

informe.append(
    f"Ciudad con mayor energia media: "
    f"{ciudad_mayor_energia['city']}"
)

informe.append(
    f"Energia media: "
    f"{ciudad_mayor_energia['energia_media_kwh']:.2f} kWh"
)

informe.append("\nRESULTADOS POR CONECTOR")
informe.append("-" * 30)

informe.append(
    f"Conector con mayor energia media: "
    f"{conector_mayor_energia['connector_type']}"
)

informe.append(
    f"Energia media: "
    f"{conector_mayor_energia['energia_media_kwh']:.2f} kWh"
)

informe.append("\nCALIDAD DE LOS DATOS")
informe.append("-" * 30)

informe.append(
    f"Registros con posibles anomalias: "
    f"{int(df_analisis['posible_anomalia'].sum())}"
)

informe.append("\nNota: los resultados no permiten medir directamente")
informe.append("la eficiencia energetica ni la autonomia de los vehiculos.")

# Guardar informe
ruta_informe = DATA_DIR / 'informe_resultados.txt'

ruta_informe.write_text(
    "\n".join(informe),
    encoding='utf-8'
)

print("Informe guardado en:", ruta_informe)
print("\n".join(informe))


Informe guardado en: datos/informe_resultados.txt
EA4 - ANALISIS DE RECARGA DE VEHICULOS ELECTRICOS

Total de registros analizados: 5000
Ciudades analizadas: 7
Estaciones analizadas: 890
Tipos de conectores: 4

RESULTADOS TEMPORALES
------------------------------
Hora de mayor actividad: 0:00
Sesiones registradas: 256

RESULTADOS POR CIUDAD
------------------------------
Ciudad con mayor energia media: Amsterdam
Energia media: 74.24 kWh

RESULTADOS POR CONECTOR
------------------------------
Conector con mayor energia media: Tesla Supercharger
Energia media: 75.03 kWh

CALIDAD DE LOS DATOS
------------------------------
Registros con posibles anomalias: 1691

Nota: los resultados no permiten medir directamente
la eficiencia energetica ni la autonomia de los vehiculos.


In [212]:
# Seleccionar sesiones de Ámsterdam mediante NumPy
mascara_amsterdam = (
    df_analisis['city'].to_numpy() == 'Amsterdam'
)

energia_numpy = np.sum(
    df_analisis.loc[mascara_amsterdam, 'energy_consumed_kwh']
)

energia_groupby = analisis_ciudades.loc[
    analisis_ciudades['city'] == 'Amsterdam',
    'energia_total_kwh'
].iloc[0]

print("Resultado NumPy:", round(energia_numpy, 2))
print("Resultado groupby:", round(energia_groupby, 2))

print("¿Coinciden?:", np.isclose(
    energia_numpy,
    energia_groupby
))

Resultado NumPy: 51148.63
Resultado groupby: 51148.63
¿Coinciden?: True


### Verificación de resultados mediante NumPy y Pandas

Para comprobar la fiabilidad de las operaciones realizadas, se verificó el cálculo de la energía total consumida en las sesiones registradas en Ámsterdam mediante dos procedimientos independientes.

En primer lugar, se utilizó una máscara booleana de NumPy para seleccionar los registros correspondientes a esta ciudad y calcular la suma de energía consumida. Posteriormente, se comparó el resultado con el valor obtenido mediante la agrupación de datos con `groupby()` de Pandas.

Ambos procedimientos devolvieron **51.148,63 kWh**. La función `np.isclose()` confirmó que los resultados coinciden (`True`).

Esta comprobación permite validar que la selección de registros y la agregación de energía se están realizando correctamente para este caso de prueba.

In [213]:

%%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

# Configuración
st.set_page_config(
    page_title="EA4 - Recarga de vehículos eléctricos",
    page_icon="⚡",
    layout="wide"
)

st.title("⚡ Dashboard de recarga de vehículos eléctricos")
st.markdown(
    "Análisis del consumo energético por ciudad, "
    "hora y tipo de conector"
)

# Cargar datos
RUTA = Path("datos/ev_charging_telemetry.csv")

@st.cache_data
def cargar_datos():
    datos = pd.read_csv(RUTA)
    datos["timestamp"] = pd.to_datetime(
        datos["timestamp"], errors="coerce"
    )
    datos = datos.dropna(subset=["timestamp"]).copy()

    datos["hora"] = datos["timestamp"].dt.hour

    datos["potencia_media_observada_kw"] = (
        datos["energy_consumed_kwh"] /
        (datos["session_duration_mins"] / 60)
    )

    datos["ratio_potencia"] = (
        datos["potencia_media_observada_kw"] /
        datos["power_kw"]
    )

    return datos

df = cargar_datos()

# Filtros laterales
st.sidebar.header("Filtros")

ciudades = st.sidebar.multiselect(
    "Seleccionar ciudades",
    sorted(df["city"].unique()),
    default=sorted(df["city"].unique())
)

conectores = st.sidebar.multiselect(
    "Seleccionar conectores",
    sorted(df["connector_type"].unique()),
    default=sorted(df["connector_type"].unique())
)

fecha_min = df["timestamp"].min().date()
fecha_max = df["timestamp"].max().date()

periodo = st.sidebar.date_input(
    "Seleccionar periodo",
    value=(fecha_min, fecha_max),
    min_value=fecha_min,
    max_value=fecha_max
)

if len(periodo) == 2:
    inicio, fin = periodo

    filtrado = df[
        (df["city"].isin(ciudades)) &
        (df["connector_type"].isin(conectores)) &
        (df["timestamp"].dt.date >= inicio) &
        (df["timestamp"].dt.date <= fin)
    ].copy()
else:
    filtrado = df.iloc[0:0].copy()

if filtrado.empty:
    st.warning("No hay registros para los filtros seleccionados.")
    st.stop()

# Indicadores principales
col1, col2, col3, col4 = st.columns(4)

col1.metric(
    "Sesiones",
    f"{len(filtrado):,}"
)

col2.metric(
    "Energía total",
    f"{filtrado['energy_consumed_kwh'].sum():,.2f} kWh"
)

col3.metric(
    "Energía media",
    f"{filtrado['energy_consumed_kwh'].mean():.2f} kWh"
)

col4.metric(
    "Potencia media observada",
    f"{filtrado['potencia_media_observada_kw'].mean():.2f} kW"
)

st.divider()

# Gráficos
st.subheader("1. Sesiones por hora")

horario = (
    filtrado.groupby("hora")
    .size()
    .reset_index(name="numero_sesiones")
)

fig1 = px.bar(
    horario,
    x="hora",
    y="numero_sesiones",
    title="Número de sesiones por hora"
)

st.plotly_chart(fig1, use_container_width=True)

col_a, col_b = st.columns(2)

with col_a:
    st.subheader("2. Energía media por ciudad")

    ciudades_df = (
        filtrado.groupby("city")["energy_consumed_kwh"]
        .mean()
        .reset_index()
    )

    fig2 = px.bar(
        ciudades_df,
        x="city",
        y="energy_consumed_kwh",
        title="Energía media por ciudad"
    )

    st.plotly_chart(fig2, use_container_width=True)

with col_b:
    st.subheader("3. Potencia por conector")

    conectores_df = (
        filtrado.groupby("connector_type")
        ["potencia_media_observada_kw"]
        .mean()
        .reset_index()
    )

    fig3 = px.bar(
        conectores_df,
        x="connector_type",
        y="potencia_media_observada_kw",
        title="Potencia media observada por conector"
    )

    st.plotly_chart(fig3, use_container_width=True)

# Relación duración y energía
st.subheader("4. Duración y energía consumida")

fig4 = px.scatter(
    filtrado,
    x="session_duration_mins",
    y="energy_consumed_kwh",
    color="connector_type",
    title="Relación entre duración y energía"
)

st.plotly_chart(fig4, use_container_width=True)

# Tabla de datos
with st.expander("Consultar registros filtrados"):
    st.dataframe(filtrado, use_container_width=True)

st.caption(
    "Fuente: Global EV Charging Station Telemetry - Hugging Face"
)


Overwriting app.py
